# E-Commerce A/B Test Analysis: Pricing Strategy
**Google Colab project based on the supplied report and CSV datasets**

Objective: test whether the new landing page (`treatment`) improves conversion compared with the existing page (`control`).

The notebook cleans the data, compares conversion rates, runs Welch's t-test, chi-square and ANOVA tests, visualizes results, and generates a business recommendation. Significance level: **α = 0.05**.

## 1. Import libraries

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 50)
print("Libraries imported successfully.")

## 2. Load files

In [ ]:
# Upload both ab_data.csv and countries.csv when prompted.
if not (os.path.exists("ab_data.csv") and os.path.exists("countries.csv")):
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))

ab = pd.read_csv("ab_data.csv")
countries = pd.read_csv("countries.csv")
print("A/B data shape:", ab.shape)
print("Countries data shape:", countries.shape)
print("A/B columns:", list(ab.columns))
print("Country columns:", list(countries.columns))
display(ab.head())
display(countries.head())

## 3. Clean and validate data

Following the report: remove group/page mismatches, retain one row per user, then merge country information. Actual cleaning counts are calculated from the files.

In [ ]:
required_ab = {"user_id", "group", "landing_page", "converted"}
required_country = {"user_id", "country"}
if required_ab - set(ab.columns):
    raise ValueError(f"Missing A/B columns: {required_ab - set(ab.columns)}")
if required_country - set(countries.columns):
    raise ValueError(f"Missing country columns: {required_country - set(countries.columns)}")

valid_pair = (
    ((ab["group"] == "control") & (ab["landing_page"] == "old_page")) |
    ((ab["group"] == "treatment") & (ab["landing_page"] == "new_page"))
)
mismatch_count = int((~valid_pair).sum())
clean = ab.loc[valid_pair].copy()
duplicate_rows = int(clean["user_id"].duplicated(keep="first").sum())
clean = clean.drop_duplicates(subset="user_id", keep="first").copy()
clean["converted"] = pd.to_numeric(clean["converted"], errors="coerce")
clean = clean[clean["converted"].isin([0, 1])].copy()
clean["converted"] = clean["converted"].astype(int)

countries_clean = countries.drop_duplicates(subset="user_id", keep="first").copy()
data = clean.merge(countries_clean, on="user_id", how="inner", validate="one_to_one")
data = data.dropna(subset=["group", "country", "converted"]).copy()

print(f"Original rows: {len(ab):,}")
print(f"Group/page mismatches removed: {mismatch_count:,}")
print(f"Duplicate user rows removed after mismatch filtering: {duplicate_rows:,}")
print(f"Rows after merging country data: {len(data):,}")
print("\nUsers by group:")
display(data["group"].value_counts().to_frame("users"))
print("\nUsers by country:")
display(data["country"].value_counts().to_frame("users"))

## 4. Conversion-rate summary

In [ ]:
summary = data.groupby("group").agg(
    users=("user_id", "nunique"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
).reindex(["control", "treatment"])
summary["conversion_rate_pct"] = summary["conversion_rate"] * 100
display(summary)

control_rate = summary.loc["control", "conversion_rate"]
treatment_rate = summary.loc["treatment", "conversion_rate"]
difference = treatment_rate - control_rate
relative_lift = difference / control_rate * 100 if control_rate else np.nan
print(f"Control rate: {control_rate:.3%}")
print(f"Treatment rate: {treatment_rate:.3%}")
print(f"Difference (treatment - control): {difference*100:.3f} percentage points")
print(f"Relative lift: {relative_lift:.2f}%")

## 5. Plot conversion rates with 95% confidence intervals

In [ ]:
labels, rates, errors = [], [], []
for group in ["control", "treatment"]:
    values = data.loc[data["group"] == group, "converted"]
    p, n = values.mean(), len(values)
    se = np.sqrt(p * (1-p) / n)
    labels.append(group)
    rates.append(p * 100)
    errors.append(1.96 * se * 100)

plt.figure(figsize=(7, 5))
bars = plt.bar(labels, rates, yerr=errors, capsize=6)
plt.ylabel("Conversion rate (%)")
plt.title("Conversion Rate by Pricing Strategy (95% CI)")
plt.ylim(0, max(rates) + max(errors) + 1)
for bar, rate in zip(bars, rates):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.12,
             f"{rate:.2f}%", ha="center")
plt.tight_layout()
plt.show()

## 6. Test 1 — Welch's independent-samples t-test

**H₀:** control and treatment have equal mean conversion. **H₁:** their mean conversion differs.

In [ ]:
control = data.loc[data["group"] == "control", "converted"]
treatment = data.loc[data["group"] == "treatment", "converted"]
t_stat, t_p = stats.ttest_ind(treatment, control, equal_var=False)

n_t, n_c = len(treatment), len(control)
mean_t, mean_c = treatment.mean(), control.mean()
var_t, var_c = treatment.var(ddof=1), control.var(ddof=1)
se = np.sqrt(var_t/n_t + var_c/n_c)
df = (var_t/n_t + var_c/n_c)**2 / ((var_t/n_t)**2/(n_t-1) + (var_c/n_c)**2/(n_c-1))
crit = stats.t.ppf(0.975, df)
ci_low, ci_high = (mean_t-mean_c)-crit*se, (mean_t-mean_c)+crit*se

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {t_p:.4f}")
print(f"Difference: {(mean_t-mean_c)*100:.3f} percentage points")
print(f"95% CI: [{ci_low*100:.3f}, {ci_high*100:.3f}] percentage points")
print("Decision:", "Reject H0" if t_p < 0.05 else "Fail to reject H0")

## 7. Test 2 — Chi-square test of independence

**H₀:** strategy and conversion are independent. **H₁:** they are associated.

In [ ]:
contingency = pd.crosstab(data["group"], data["converted"]).reindex(
    index=["control", "treatment"], columns=[0, 1], fill_value=0
)
contingency.columns = ["Not converted", "Converted"]
display(contingency)
chi2, chi_p, chi_df, expected = stats.chi2_contingency(contingency.values)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {chi_df}")
print(f"p-value: {chi_p:.4f}")
print("Decision:", "Reject H0" if chi_p < 0.05 else "Fail to reject H0")

## 8. Test 3 — One-way ANOVA

Compare conversion across countries and across the six country/strategy combinations. Because conversion is binary, treat ANOVA as a supplementary comparison.

In [ ]:
country_groups = [g["converted"].to_numpy() for _, g in data.groupby("country") if len(g)]
f_country, p_country = stats.f_oneway(*country_groups)

data["country_strategy"] = data["country"].astype(str) + " - " + data["group"].astype(str)
combo_groups = [g["converted"].to_numpy() for _, g in data.groupby("country_strategy") if len(g)]
f_combo, p_combo = stats.f_oneway(*combo_groups)

country_summary = data.groupby("country").agg(
    users=("user_id", "nunique"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
)
country_summary["conversion_rate_pct"] = country_summary["conversion_rate"] * 100
display(country_summary)

anova_results = pd.DataFrame([
    {"Comparison": "Conversion across countries", "F-statistic": f_country, "p-value": p_country},
    {"Comparison": "Country × strategy combinations", "F-statistic": f_combo, "p-value": p_combo}
])
display(anova_results)

## 9. Country and strategy comparison

In [ ]:
country_strategy_summary = data.groupby(["country", "group"]).agg(
    users=("user_id", "nunique"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
).reset_index()
country_strategy_summary["conversion_rate_pct"] = country_strategy_summary["conversion_rate"] * 100
display(country_strategy_summary)

pivot = country_strategy_summary.pivot(index="country", columns="group", values="conversion_rate_pct")
pivot.reindex(columns=["control", "treatment"]).plot(kind="bar", figsize=(8, 5), rot=0)
plt.ylabel("Conversion rate (%)")
plt.xlabel("Country")
plt.title("Conversion Rate by Country and Strategy")
plt.legend(title="Strategy")
plt.tight_layout()
plt.show()

## 10. Automated conclusion and recommendation

In [ ]:
results_df = pd.DataFrame([
    {"Test": "Welch's t-test", "p-value": t_p},
    {"Test": "Chi-square test", "p-value": chi_p},
    {"Test": "ANOVA: countries", "p-value": p_country},
    {"Test": "ANOVA: country × strategy", "p-value": p_combo}
])
results_df["Significant at alpha = 0.05"] = results_df["p-value"] < 0.05
display(results_df)

if t_p < 0.05 and treatment_rate > control_rate:
    conclusion = "Treatment conversion is higher, and the difference is statistically significant."
    recommendation = "Consider rolling out the new page after checking business impact and experiment quality."
elif t_p < 0.05 and treatment_rate < control_rate:
    conclusion = "Treatment conversion is lower, and the difference is statistically significant."
    recommendation = "Retain the existing page; treatment performed significantly worse."
else:
    conclusion = "The data does not provide statistically significant evidence of a conversion-rate difference."
    recommendation = "Retain the existing page for now. If a small improvement is important, run a follow-up test designed to detect smaller effects."

print("CONCLUSION:", conclusion)
print("RECOMMENDATION:", recommendation)
print("Note: failure to find significance does not prove the pages are identical.")

## 11. Export results (optional)

In [ ]:
data.to_csv("ab_test_cleaned_data.csv", index=False)
summary.to_csv("ab_test_group_summary.csv")
country_strategy_summary.to_csv("ab_test_country_strategy_summary.csv", index=False)
results_df.to_csv("ab_test_statistical_test_results.csv", index=False)
print("Exported cleaned data and summary CSV files. Download them from the Colab Files panel.")